# Pre-trained Model: EfficientNetV2-B0

Transfer learning for dish recognition.

**Input:** images as `data/train/<class>/` and `data/val/<class>/`
**Output:** `artifacts/model.pth` + `artifacts/config.json`
**Backbone:** `tf_efficientnetv2_b0` from timm, ImageNet weights

In [3]:
%pip install pillow timm torch torchvision

  Using cached pillow-12.3.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (9.1 kB)
  Using cached timm-1.0.29-py3-none-any.whl.metadata (43 kB)
  Using cached torch-2.14.0-cp312-cp312-macosx_14_0_arm64.whl.metadata (38 kB)
  Using cached torchvision-0.29.0-cp312-cp312-macosx_14_0_arm64.whl.metadata (5.7 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached pillow-12.3.0-cp312-cp312-macosx_11_0_arm64.whl (4.8 MB)
Using cached timm-1.0.29-py3-none-any.whl (2.6 MB)
Using cached torch-2.14.0-cp312-cp312-macosx_14_0_arm64.whl (127.3 MB)
Using cached torchvision-0.29.0-cp312-cp312-macosx_14_0_arm64.whl (1.8 MB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (53

In [4]:
# Standard library
import json
import time
from pathlib import Path

# Numerics & images
import numpy as np
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# timm
import timm
from timm.data import resolve_data_config, create_transform

In [5]:
DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"torch       {torch.__version__}")
print(f"timm        {timm.__version__}")
print(f"device      {DEVICE}")

torch       2.14.0
timm        1.0.29
device      mps


In [6]:
MODEL_NAME = "tf_efficientnetv2_b0"
DATA_DIR = Path("../../data")      # relative to notebooks/andreas/
ARTIFACTS_DIR = Path("../../artifacts")

BATCH_SIZE = 32
EPOCHS_HEAD = 8      # Phase 1: classification head only
EPOCHS_FINETUNE = 6  # Phase 2: top blocks unfrozen
LR_HEAD = 1e-3
LR_FINETUNE = 1e-5   # 100x smaller, otherwise you destroy the weights
LABEL_SMOOTHING = 0.1
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0)
cfg = timm.data.resolve_model_data_config(model)
print(cfg)

model.safetensors: reconstructing file:   0%|          |  0.00B / 28.8MB            

model.safetensors: downloading bytes:           |  0.00B            

{'input_size': (3, 192, 192), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.875, 'crop_mode': 'center'}


In [8]:
train_tf = create_transform(**cfg, is_training=True, auto_augment="rand-m9-mstd0.5")
val_tf = create_transform(**cfg, is_training=False)
print(val_tf)

Compose(
    Resize(size=219, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(192, 192))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)
